# Day 059 — Exercise 1: FastAPI BackgroundTasks

FastAPI has a built-in `BackgroundTasks` mechanism: inject it as a route parameter, call `background_tasks.add_task(fn, *args)`, and FastAPI runs `fn` **after** the HTTP response is sent — so the client gets an immediate 202 response.

**Test insight:** `TestClient` runs `BackgroundTasks` synchronously before returning the response object — so the job is already done by the time you read `r.json()`. No polling or sleeping needed in tests.

In [ ]:
import secrets
from fastapi import BackgroundTasks, FastAPI, HTTPException
from pydantic import BaseModel, Field
from starlette.testclient import TestClient


## Task

Implement `build_background_tasks_api(process_fn=None)` — return a FastAPI app with:

```
POST /run   {"text": "..."}  → 202  {"job_id": "abc123", "status": "pending"}
GET  /jobs/{job_id}          → 200  {"status": "done", "result": "..."}
                             → 404  if job_id unknown
```

- Use `background_tasks: BackgroundTasks` as a route parameter (FastAPI injects it)
- If `process_fn` is provided: use it instead of the default (uppercase)
- Store results in a `dict` keyed by `job_id`
- Return 422 automatically when `text` is empty (`Field(min_length=1)`)

## Your Implementation

In [ ]:
def build_background_tasks_api(process_fn=None) -> FastAPI:
    """Return a FastAPI app using BackgroundTasks for deferred processing.

    Endpoints:
      POST /run   body: {"text": "..."}  → 202 + {"job_id": "...", "status": "pending"}
      GET /jobs/{job_id}               → {"status": "done", "result": "..."}
                                         or 404 if unknown

    process_fn: optional callable(text: str) -> str for testing.
                If None, uppercase the text (trivial default).

    Key insight: TestClient runs BackgroundTasks synchronously, so the job
    is already done by the time the POST response is received in tests.
    """
    # TODO: create app, _results dict, POST /run with background_tasks param,
    #       GET /jobs/{job_id}
    raise NotImplementedError


In [ ]:
def build_background_tasks_api(process_fn=None) -> FastAPI:
    app = FastAPI()
    _results: dict = {}

    class _RunReq(BaseModel):
        text: str = Field(min_length=1)

    @app.post("/run", status_code=202)
    def run_task(req: _RunReq, background_tasks: BackgroundTasks):
        job_id = secrets.token_hex(4)
        _results[job_id] = {"status": "pending"}

        def process():
            try:
                result = process_fn(req.text) if process_fn else req.text.upper()
                _results[job_id] = {"status": "done", "result": result}
            except Exception as e:
                _results[job_id] = {"status": "error", "error": str(e)}

        background_tasks.add_task(process)
        return {"job_id": job_id, "status": "pending"}

    @app.get("/jobs/{job_id}")
    def get_job(job_id: str):
        if job_id not in _results:
            raise HTTPException(404, "Job not found")
        return _results[job_id]

    return app


## Automated checks

In [ ]:
score, total = 0, 5
try:
    app    = build_background_tasks_api(process_fn=str.upper)
    client = TestClient(app, raise_server_exceptions=False)

    # TestClient runs BackgroundTasks synchronously before returning response
    r = client.post("/run", json={"text": "hello"})
    assert r.status_code == 202, f"Expected 202, got {r.status_code}"
    score += 1; print("\u2705 POST /run returns 202")

    body = r.json()
    assert "job_id" in body, f"Expected job_id in response: {body}"
    score += 1; print("\u2705 response contains job_id")

    # BackgroundTask already completed (TestClient is sync)
    job_id = body["job_id"]
    r2 = client.get(f"/jobs/{job_id}")
    assert r2.status_code == 200, f"Expected 200, got {r2.status_code}"
    data = r2.json()
    assert data["status"] == "done", f"Expected done, got {data}"
    assert data.get("result") == "HELLO", f"Expected 'HELLO', got {data.get('result')}"
    score += 1; print("\u2705 job result is available after POST (TestClient sync)")

    # unknown job → 404
    r3 = client.get("/jobs/unknown_id")
    assert r3.status_code == 404, f"Expected 404, got {r3.status_code}"
    score += 1; print("\u2705 unknown job returns 404")

    # empty text → 422
    r4 = client.post("/run", json={"text": ""})
    assert r4.status_code == 422, f"Expected 422 for empty text, got {r4.status_code}"
    score += 1; print("\u2705 empty text returns 422")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def build_background_tasks_api(process_fn=None) -> FastAPI:
    app = FastAPI()
    _results: dict = {}

    class _RunReq(BaseModel):
        text: str = Field(min_length=1)

    @app.post("/run", status_code=202)
    def run_task(req: _RunReq, background_tasks: BackgroundTasks):
        job_id = secrets.token_hex(4)
        _results[job_id] = {"status": "pending"}

        def process():
            try:
                result = process_fn(req.text) if process_fn else req.text.upper()
                _results[job_id] = {"status": "done", "result": result}
            except Exception as e:
                _results[job_id] = {"status": "error", "error": str(e)}

        background_tasks.add_task(process)
        return {"job_id": job_id, "status": "pending"}

    @app.get("/jobs/{job_id}")
    def get_job(job_id: str):
        if job_id not in _results:
            raise HTTPException(404, "Job not found")
        return _results[job_id]

    return app
```

**Why it works:** `background_tasks.add_task(process)` registers `process` to run after the response is sent. With TestClient, it runs synchronously, so the result is available immediately. In production, it runs in the same thread pool as the ASGI server after the response bytes are flushed.

</details>